# Weekly benchmarks

Parse the weekly ClimaCoupler and ClimaLand Buildkite jobs mirrored by the
`fetch-weekly-buildkite` stage and plot their scalar metrics over time.

The coupler runs are those reported into the CliMA Slack `#coupler-report`
channel: the target AMIP configuration (`climacoupler-amip`), the validation
suite (`climacoupler-longruns`), and the CPU/GPU comparison
(`climacoupler-cpu-gpu-benchmarks`). `climaland-long-runs` runs twice weekly,
once as a ~2 year run and once as a 19/20 year run.

Accuracy comes from the leaderboard RMSE CSVs the long runs have written
as Buildkite artifacts since 2026-11-09 (ClimaLand.jl#1950,
ClimaCoupler.jl#2206), which the same stage mirrors. Together these give
speed, stability, and accuracy for each configuration over time.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    print("Moving to repo root")
    os.chdir("..")

Moving to repo root


## Log parsing

Every metric below comes out of the job's raw log. Buildkite wraps output in
ANSI colour codes and OSC timestamp markers, so those are stripped first.

Note that ClimaCoupler's progress format changed around 2026-07-22. Older
logs print a block of `key = value` lines with `simulation_time` as a
human-readable duration (`"156 weeks, 6 days"`) alongside
`n_steps_completed` and `percent_complete`; newer ones print
`time = <date> (<elapsed>)` and `step = N (P%)`. Both are handled so the
series is continuous across the change.

In [2]:
import glob
import json
import math
import re
from datetime import datetime, timezone

import pandas as pd

ANSI = re.compile(r"\x1b\[[0-9;]*[a-zA-Z]|\x1b[_\]][^\x07]*\x07")
BK_TIMESTAMP = re.compile(r"^_bk;t=\d+", re.M)

# Units Julia prints in elapsed-time summaries like "1 y 354 d"
UNIT_DAYS = {"y": 365.0, "d": 1.0, "h": 1 / 24, "m": 1 / 1440, "s": 1 / 86400}


def clean_log(text: str) -> str:
    """Strip colour codes and Buildkite's per-line timestamp markers."""
    return BK_TIMESTAMP.sub("", ANSI.sub("", text))


def parse_elapsed_days(text: str) -> float | None:
    """'300 d' / '1 d 1 h' / '1 y 354 d' -> float days."""
    days = 0.0
    for value, unit in re.findall(r"([0-9.]+)\s*(y|d|h|m|s)\b", text):
        days += float(value) * UNIT_DAYS[unit]
    return days or None


# Units Julia's canonicalized durations use in the pre-2026-07-22 progress
# block, e.g. "156 weeks, 6 days". Months and years never appear, since
# neither is a fixed-length period.
CANONICAL_UNIT_DAYS = {
    "week": 7.0,
    "day": 1.0,
    "hour": 1 / 24,
    "minute": 1 / 1440,
    "second": 1 / 86400,
    "millisecond": 1 / 86_400_000,
    "microsecond": 1 / 86_400_000_000,
}


def parse_canonical_duration_days(text: str) -> float | None:
    """'156 weeks, 6 days' / '1 day, 10 hours' -> float days."""
    days = 0.0
    pattern = r"([0-9.]+)\s*(" + "|".join(CANONICAL_UNIT_DAYS) + r")s?\b"
    for value, unit in re.findall(pattern, text):
        days += float(value) * CANONICAL_UNIT_DAYS[unit]
    return days or None


def parse_duration_days(text: str | None) -> float | None:
    """Config durations like '732days' or '120secs' -> float days."""
    if not text:
        return None
    match = re.match(r"([0-9.]+)\s*(years?|days?|hours?|mins?|secs?)", text)
    if not match:
        return None
    return float(match.group(1)) * UNIT_DAYS[match.group(2)[0]]


def parse_run_metrics(raw_log: str) -> dict:
    """Extract the scalar metrics from one ClimaCoupler run log."""
    log = clean_log(raw_log)
    out: dict = {}

    # Throughput. ClimaCoupler runs print a final "Info: SYPD:" summary
    # line, which is the authoritative figure. ClimaLand runs never do --
    # they only report "estimated_sypd" inside each progress block -- so
    # fall back to the last finite one of those. Early blocks legitimately
    # read "Inf" before enough steps have elapsed to estimate, so those are
    # skipped rather than treated as a parse failure.
    sypd = re.findall(r"\bInfo: SYPD:\s*([0-9.eE+-]+)", log)
    if sypd:
        out["sypd"] = float(sypd[-1])
    else:
        out["sypd"] = None
        for value in re.findall(r'estimated_sypd = "([^"]*)"', log):
            try:
                number = float(value)
            except ValueError:
                continue
            if math.isfinite(number):
                out["sypd"] = number
    wps = re.findall(r"Walltime per coupling step:\s*([0-9.eE+-]+)", log)
    out["walltime_per_step_s"] = float(wps[-1]) if wps else None

    # How far the simulation got. For a crashed run this is the last
    # progress report before the exception, i.e. the run length before
    # the crash.
    progress = re.findall(
        r"Info: Progress\s*\n?\s*│?\s*time = (\S+) \(([^)]*)\)"
        r"\s*\n?\s*│?\s*step = (\d+) \(([0-9.]+)%\)",
        log,
    )
    if progress:
        sim_date, elapsed, step, percent = progress[-1]
        out["last_sim_date"] = sim_date
        out["last_sim_days"] = parse_elapsed_days(elapsed)
        out["last_step"] = int(step)
        out["percent_complete"] = float(percent)
    else:
        # Pre-2026-07-22 progress format
        out["last_sim_date"] = None
        sim_time = re.findall(r'simulation_time = "([^"]*)"', log)
        out["last_sim_days"] = (
            parse_canonical_duration_days(sim_time[-1]) if sim_time else None
        )
        steps = re.findall(r"n_steps_completed = (\d+)", log)
        out["last_step"] = int(steps[-1]) if steps else None
        percent = re.findall(r'percent_complete = "([0-9.]+)%"', log)
        out["percent_complete"] = float(percent[-1]) if percent else None

    # Config knobs. A run prints several config dicts (the coupler's, then
    # the atmos one it derives); only the coupler dict carries "job_id", so
    # key off that line to be sure t_end/dt/h_elem come from the same dict.
    config_line = ""
    for line in log.splitlines():
        if '"job_id" =>' in line and "Dict{" in line:
            config_line = line
            break
    match = re.search(r'"job_id" => "([^"]+)"', config_line)
    out["config_job_id"] = match.group(1) if match else None
    for key, cast in [
        ("t_end", str),
        ("dt", str),
        ("h_elem", int),
        ("z_elem", int),
        ("FLOAT_TYPE", str),
    ]:
        match = re.search(rf'"{key}" => "?([A-Za-z0-9_.]+)"?', config_line)
        out[key.lower()] = cast(match.group(1)) if match else None

    # Fraction of the configured run length actually simulated. This is the
    # crash-severity metric, and unlike raw simulated days it is comparable
    # across configs with different t_end. Prefer the percentage the model
    # reports itself; fall back to the ratio when a log predates it or the
    # progress block was cut off.
    out["t_end_days"] = parse_duration_days(out.get("t_end"))
    if out["percent_complete"] is not None:
        out["fraction_completed"] = out["percent_complete"] / 100.0
    elif out["last_sim_days"] and out["t_end_days"]:
        out["fraction_completed"] = out["last_sim_days"] / out["t_end_days"]
    else:
        out["fraction_completed"] = None

    # Crash signature: the first Julia-level error, which distinguishes
    # e.g. a fraction assertion from a NaN blowing up in a GPU kernel.
    match = re.search(r"^ERROR: (?:LoadError: )?(.{0,160})", log, re.M)
    out["error"] = match.group(1).strip() if match else None
    return out

## Build the table

In [3]:
EMOJI_TOKEN = re.compile(r":[a-z0-9_+-]+:")


def clean_job_name(name: str | None) -> str | None:
    """Strip Buildkite's ``:emoji:`` tokens from a job label."""
    if not name:
        return None
    return EMOJI_TOKEN.sub("", name).strip() or None


def parse_iso(text: str | None) -> datetime | None:
    if not text:
        return None
    dt = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


rows = []
for path in sorted(glob.glob("data/buildkite-weekly/*/*.json")):
    with open(path) as f:
        job = json.load(f)
    started = parse_iso(job.get("job_started_at") or job.get("job_created_at"))
    finished = parse_iso(job.get("job_finished_at"))
    row = {
        "date": started,
        "pipeline": job["pipeline"],
        "build_number": job["build_number"],
        "build_state": job["build_state"],
        "job_id": job["job_id"],
        "job_name": job["job_name"],
        "job_state": job["job_state"],
        "commit": job.get("commit"),
        "job_hours": (
            (finished - started).total_seconds() / 3600
            if started and finished
            else None
        ),
    }
    row.update(parse_run_metrics(job["raw_log_txt"]))
    # Identify the configuration. ClimaCoupler names it in its config dict;
    # ClimaLand has no equivalent, and its Buildkite job label is what
    # actually distinguishes the configs ("Soil" vs "Soil, 20 years"), so
    # fall back to that. It also rescues any coupler job whose config dict
    # did not parse.
    row["run_name"] = row["config_job_id"] or clean_job_name(row["job_name"])
    rows.append(row)

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

# A job that passed ran to completion by definition. Its last progress block
# is only the final checkpoint before the end -- the reporting interval
# differs per model (ClimaLand prints every 10%, the coupler every few
# percent) -- so the raw figure under-reports, and roughly half of all passed
# jobs would otherwise look like they stopped short. Pin them to 1.0 so this
# column means "how far did it get" rather than "when was progress last
# printed". Failed and cancelled jobs keep their parsed value, which is the
# number of interest for them.
df.loc[df["job_state"] == "passed", "fraction_completed"] = 1.0

# Simulated years actually reached. For a run that passed, the configured
# length is what it simulated, for the same reason fraction_completed is
# pinned above: the last progress block is only the final checkpoint. Land
# runs have no parsed t_end, so they fall back to the last reported value.
df["simulated_years"] = df["last_sim_days"] / 365.0
completed = (df["job_state"] == "passed") & df["t_end_days"].notna()
df.loc[completed, "simulated_years"] = df.loc[completed, "t_end_days"] / 365.0

print(f"{len(df)} jobs, {df['date'].min()} to {df['date'].max()}")
df["run_name"].value_counts()

1269 jobs, 2025-09-07 05:11:03.887000+00:00 to 2026-11-18 03:44:09.804000+00:00


run_name
amip                                             64
cmip_edonly_land                                 58
cmip_edonly_bucket                               58
slabplanet_evolve_ocean                          47
slabplanet_aqua_evolve_ocean                     46
                                                 ..
GPU AMIP + prog. EDMF + 1M + integrated land      1
GPU CMIP: diag. EDMF + integrated land            1
GPU CMIP: prog. EDMF + integrated land            1
Current AMIP: diagedmf + 0M + integrated land     1
Aquaplanet: evolving slab ocean                   1
Name: count, Length: 66, dtype: int64

In [4]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/weekly-benchmarks.csv", index=False)
df.tail(10)[
    ["date", "pipeline", "run_name", "job_state", "sypd", "fraction_completed"]
]

,date,pipeline,run_name,job_state,sypd,fraction_completed
1259,2026-11-15 07:32:15.204+00:00,climacoupler-longruns,cmip_edonly_bucket,passed,1.429161,1.0
1260,2026-11-15 08:27:25.828+00:00,climacoupler-longruns,cmip_edonly_land,passed,1.380743,1.0
1261,2026-11-15 08:46:31.358+00:00,climacoupler-longruns,cmip_progedmf_land,passed,1.240252,1.0
1262,2026-11-17 04:02:10.826+00:00,climaland-long-runs,"Snowy Land, 19 years",passed,134.131000,1.0
1263,2026-11-17 04:04:42.766+00:00,climaland-long-runs,"Snowy Land (Optimal LAI), 19 years",passed,105.304000,1.0
1264,2026-11-17 04:10:55.146+00:00,climaland-long-runs,"Soil, 20 years",passed,185.946000,1.0
1265,2026-11-18 03:30:01.890+00:00,climaland-long-runs,Snowy Land + P Model,passed,155.255000,1.0
1266,2026-11-18 03:30:05.804+00:00,climaland-long-runs,Soil,passed,366.986000,1.0
1267,2026-11-18 03:42:47.771+00:00,climaland-long-runs,Global bucket simulation,passed,356.272000,1.0
1268,2026-11-18 03:44:09.804+00:00,climaland-long-runs,Snowy Land + P Model (Optimal LAI),passed,113.757000,1.0


## Recent failures

What actually killed each run, alongside how far it got. The signature
distinguishes e.g. a surface-fraction assertion from a NaN blowing up
inside a GPU kernel.

In [5]:
# Job states that mean the run actually fell over. "canceled" is excluded
# throughout: a human stopping a run is not a model failure.
FAILURE_STATES = ["failed", "timed_out", "broken"]

failures = df[
    df["job_state"].isin(FAILURE_STATES) & df["error"].notna()
].sort_values("date", ascending=False)

failures.head(20)[
    ["date", "run_name", "job_state", "fraction_completed", "error"]
]

,date,run_name,job_state,fraction_completed,error
1258,2026-11-15 07:15:17.928+00:00,amip_progedmf_1m_land,failed,0.093,Found NaN
1254,2026-11-14 18:36:40.804+00:00,amip,failed,0.123,Found NaN
1244,2026-11-08 06:53:05.102+00:00,amip_progedmf_land,failed,0.820,Found NaN
1240,2026-11-08 06:53:04.964+00:00,amip_progedmf_1m_land,failed,0.062,Found NaN
1239,2026-11-07 16:13:04.824+00:00,amip,failed,NaN,Found NaN
1230,2026-11-03 03:13:05.962+00:00,"Snowy Land, 19 years",failed,0.900,Units in the simulation-output NetCDF do not m...
1229,2026-11-03 03:13:05.906+00:00,"Snowy Land (Optimal LAI), 19 years",failed,0.900,Units in the simulation-output NetCDF do not m...
1186,2026-10-18 04:19:05.998+00:00,GPU ClimaAtmos with prognostic EDMF,failed,NaN,Halting simulation due to unused parameters.
1183,2026-10-17 15:14:03.865+00:00,amip,failed,0.615,Found NaN
1175,2026-10-11 09:34:50.013+00:00,cmip_edonly_land,failed,0.410,AssertionError: minimum((ice_fraction .+ land_...


## Leaderboards

Each build's `*leaderboard_rmse.csv` artifacts, joined to the job that wrote
them into one long table: one row per job, variable, season, and data
source.

The two models write different columns. ClimaCoupler compares CliMA's RMSE
for `pr`, `rlut`, and `rsut` against the CMIP6 models' (`rmse`,
`other_models_median_rmse`, `other_models_best_rmse`, `n_other_models`).
ClimaLand gives RMSE and bias against each data source (`data_source`,
`rmse`, `bias`): ERA5 surface energy fluxes, plus carbon and vegetation
metrics for the P-model runs. Columns a model doesn't write are empty.

The run name is the one the job table above gives the same job, so the two
tables join on it. A leaderboard job whose log isn't mirrored falls back to
the coupler's run directory (`output/<run>/artifacts`, which is the config's
`job_id`) or the cleaned Buildkite label.

In [6]:
LEADERBOARD_DIR = "data/buildkite-weekly-leaderboards"
MODEL_BY_PIPELINE = {
    "climacoupler-amip": "ClimaCoupler",
    "climacoupler-longruns": "ClimaCoupler",
    "climacoupler-cpu-gpu-benchmarks": "ClimaCoupler",
    "climaland-long-runs": "ClimaLand",
}
LEADERBOARD_COLUMNS = [
    "date",
    "model",
    "pipeline",
    "build_number",
    "build_state",
    "job_id",
    "job_name",
    "job_state",
    "run_name",
    "commit",
    "artifact_path",
    "data_source",
    "short_name",
    "season",
    "rmse",
    "bias",
    "other_models_median_rmse",
    "other_models_best_rmse",
    "n_other_models",
    "units",
]

run_name_by_job = df.set_index("job_id")["run_name"].dropna().to_dict()

frames = []
for path in sorted(glob.glob(f"{LEADERBOARD_DIR}/*/*.json")):
    with open(path) as f:
        build = json.load(f)
    for artifact in build["artifacts"]:
        table = pd.read_csv(
            os.path.join(
                LEADERBOARD_DIR, build["pipeline"], artifact["local_path"]
            )
        )
        run_dir = re.match(
            r"output/([^/]+)/artifacts/", artifact["artifact_path"]
        )
        run_name = run_name_by_job.get(artifact["job_id"]) or (
            run_dir.group(1)
            if run_dir
            else clean_job_name(artifact["job_name"])
        )
        frames.append(
            table.assign(
                date=parse_iso(
                    artifact["job_started_at"] or build["build_created_at"]
                ),
                model=MODEL_BY_PIPELINE[build["pipeline"]],
                pipeline=build["pipeline"],
                build_number=build["build_number"],
                build_state=build["build_state"],
                job_id=artifact["job_id"],
                job_name=artifact["job_name"],
                job_state=artifact["job_state"],
                run_name=run_name,
                commit=build["commit"],
                artifact_path=artifact["artifact_path"],
            )
        )

leaderboards = (
    pd.concat(frames, ignore_index=True)
    .reindex(columns=LEADERBOARD_COLUMNS)
    .sort_values(["date", "run_name", "data_source", "short_name", "season"])
    .reset_index(drop=True)
)
leaderboards.to_csv("results/weekly-leaderboards.csv", index=False)
print(
    f"{len(leaderboards)} rows from {leaderboards['job_id'].nunique()} "
    f"jobs, {leaderboards['date'].min()} to {leaderboards['date'].max()}"
)
leaderboards.groupby(["model", "data_source"], dropna=False)[
    "short_name"
].unique()

485 rows from 16 jobs, 2026-11-10 03:08:06.810000+00:00 to 2026-11-18 03:44:09.804000+00:00


model         data_source              
ClimaCoupler  NaN                               [pr, rlut, rsut]
ClimaLand     ERA5                          [lhf, lwu, shf, swu]
              FlagshipCarbonMetrics               [er, gpp, nee]
              FlagshipVegetationMetrics    [fc3, ftr, lai, cveg]
Name: short_name, dtype: object

### One accuracy number per run

A time series per variable and season is too many lines to read next to
speed and stability, so each run gets one annual (`ANN`) score:

- **ClimaCoupler**: CliMA's RMSE divided by the median CMIP6 model's,
  averaged over `pr`, `rlut`, and `rsut`. 1 is the median CMIP6 model and
  lower is better. The same ratio for the best CMIP6 model is a reference.
  The CMIP6 RMSEs are from those models' `amip` experiment (5 years, the
  `cmip_model_rmse` artifact ClimaCoupler's leaderboard reads), so they're
  a like-for-like reference for the AMIP runs. The CMIP runs are compared
  against the same AMIP numbers, which is a harder bar: a coupled run also
  carries its own SST biases, which AMIP's prescribed SSTs rule out.
- **ClimaLand**: RMSE against ERA5, averaged over the four surface energy
  fluxes (`lhf`, `shf`, `lwu`, `swu`), in W m⁻². Every leaderboard run has
  these, unlike the carbon and vegetation metrics, and they share units, so
  the mean is a physical quantity. Lower is better.

A run missing any of the variables gets no score rather than a mean over
fewer of them, which would jump for reasons other than skill. Every
variable and season is in `results/weekly-leaderboards.csv`.

In [7]:
ACCURACY = {
    "ClimaCoupler": dict(data_source=None, variables=["pr", "rlut", "rsut"]),
    "ClimaLand": dict(data_source="ERA5", variables=["lhf", "lwu", "shf", "swu"]),
}

score_frames = []
for model, spec in ACCURACY.items():
    sub = leaderboards[
        (leaderboards["model"] == model)
        & (leaderboards["season"] == "ANN")
        & leaderboards["short_name"].isin(spec["variables"])
    ]
    if spec["data_source"]:
        sub = sub[sub["data_source"] == spec["data_source"]]
    sub = sub.copy()
    if model == "ClimaCoupler":
        sub["value"] = sub["rmse"] / sub["other_models_median_rmse"]
        sub["reference"] = (
            sub["other_models_best_rmse"] / sub["other_models_median_rmse"]
        )
        fmt = "{:.2f}×"
    else:
        sub["value"] = sub["rmse"]
        sub["reference"] = float("nan")
        fmt = "{:.1f}"
    for job_id, group in sub.groupby("job_id"):
        values = group.set_index("short_name")["value"]
        complete = values.reindex(spec["variables"]).notna().all()
        first = group.iloc[0]
        score_frames.append(
            {
                "date": first["date"],
                "model": model,
                "pipeline": first["pipeline"],
                "run_name": first["run_name"],
                "job_id": job_id,
                "job_state": first["job_state"],
                "accuracy": values.mean() if complete else float("nan"),
                "reference": group["reference"].mean(),
                "detail": ", ".join(
                    f"{name} {fmt.format(v)}" for name, v in values.items()
                ),
            }
        )

scores = pd.DataFrame(score_frames).sort_values("date").reset_index(drop=True)
scores[["date", "run_name", "accuracy", "detail"]]

,date,run_name,accuracy,detail
0,2026-11-10 03:08:06.810+00:00,"Snowy Land (Optimal LAI), 19 years",8.379019,"lhf 10.4, lwu 4.2, shf 10.2, swu 8.7"
1,2026-11-10 03:08:06.856+00:00,"Snowy Land, 19 years",8.050436,"lhf 10.5, lwu 3.8, shf 10.0, swu 7.9"
2,2026-11-11 03:08:06.711+00:00,Snowy Land + P Model,8.608953,"lhf 11.5, lwu 4.0, shf 10.9, swu 8.1"
3,2026-11-11 03:26:17.929+00:00,Global bucket simulation,12.850290,"lhf 12.0, lwu 14.5, shf 13.1, swu 11.8"
4,2026-11-11 03:39:56.833+00:00,Snowy Land + P Model (Optimal LAI),8.626085,"lhf 10.8, lwu 4.3, shf 10.6, swu 8.9"
5,2026-11-15 06:55:04.912+00:00,amip_progedmf_land_p2k,3.041229,"pr 1.74×, rlut 3.32×, rsut 4.06×"
6,2026-11-15 06:55:04.942+00:00,amip_progedmf_land,2.820387,"pr 1.60×, rlut 2.85×, rsut 4.02×"
7,2026-11-15 06:55:04.983+00:00,amip_edonly,2.892497,"pr 1.68×, rlut 3.63×, rsut 3.37×"
8,2026-11-15 07:32:15.204+00:00,cmip_edonly_bucket,2.802334,"pr 1.75×, rlut 3.60×, rsut 3.06×"
9,2026-11-15 08:27:25.828+00:00,cmip_edonly_land,2.766230,"pr 1.76×, rlut 3.50×, rsut 3.04×"


## Plot

Speed, stability, and accuracy for each configuration, one panel each, in a
figure per suite: the coupler's AMIP runs, its CMIP runs, its CPU/GPU
benchmarks, and the land long runs. They're different experiments with no
configurations in common, and one figure would need a legend listing all of
them. AMIP and CMIP runs both come from the coupler's longruns pipeline and
are split by run name. The CPU/GPU benchmarks don't write a leaderboard, so
their figure has no accuracy panel.

Only configurations that ran in the 21 days before their pipeline's latest
job are plotted, so retired configs drop off on their own rather than
needing a hard-coded list here. A window rather than the latest build alone
keeps runs that alternate between builds, like ClimaLand's 2-year and
19/20-year runs.

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ACTIVE_WINDOW = pd.Timedelta(days=21)

# Configs still in use, i.e. those that ran recently in their pipeline
active = set()
for pipeline, group in df.dropna(subset=["run_name"]).groupby("pipeline"):
    recent = group[group["date"] >= group["date"].max() - ACTIVE_WINDOW]
    active.update(zip(recent["pipeline"], recent["run_name"]))

# CMIP runs (coupled ocean) and AMIP runs (prescribed SSTs) share the
# longruns pipeline but are different experiments, so they're split by name.
# Older CMIP jobs whose config didn't parse are named by their Buildkite
# label, e.g. "GPU CMIP: ED only + bucket", hence the second pattern.
CMIP_RUN = re.compile(r"^cmip_|\bCMIP\b")

SUITES = [
    dict(
        slug="amip",
        title="Weekly ClimaCoupler AMIP benchmarks",
        pipelines=["climacoupler-amip", "climacoupler-longruns"],
        cmip=False,
        accuracy_title=(
            "Accuracy: RMSE relative to the median CMIP6 AMIP run (ANN)"
        ),
        accuracy_axis="RMSE ÷ CMIP6 AMIP median",
    ),
    dict(
        slug="cmip",
        title="Weekly ClimaCoupler CMIP benchmarks",
        pipelines=["climacoupler-longruns"],
        cmip=True,
        # The reference RMSEs are from CMIP6 models' amip experiment, the
        # only ones ClimaCoupler's leaderboard has, so say so on the plot
        accuracy_title=(
            "Accuracy: RMSE relative to the median CMIP6 <i>AMIP</i> run (ANN)"
        ),
        accuracy_axis="RMSE ÷ CMIP6 AMIP median",
    ),
    dict(
        slug="cpu-gpu",
        title="Weekly ClimaCoupler CPU/GPU benchmarks",
        pipelines=["climacoupler-cpu-gpu-benchmarks"],
        accuracy_title=None,
    ),
    dict(
        slug="land",
        title="Weekly ClimaLand long runs",
        pipelines=["climaland-long-runs"],
        accuracy_title="Accuracy: RMSE vs. ERA5 surface energy fluxes (ANN)",
        accuracy_axis="Mean RMSE (W m⁻²)",
    ),
]


def in_suite(suite: dict, pipeline: str, run_name: str) -> bool:
    if pipeline not in suite["pipelines"]:
        return False
    return "cmip" not in suite or suite["cmip"] == bool(
        CMIP_RUN.search(run_name)
    )


for suite in SUITES:
    suite["run_names"] = sorted(
        name for pipeline, name in active if in_suite(suite, pipeline, name)
    )
    print(f"{suite['slug']}: {', '.join(suite['run_names'])}")

amip: amip, amip_edonly, amip_progedmf_1m_land, amip_progedmf_land, amip_progedmf_land_p2k
cmip: cmip_edonly_bucket, cmip_edonly_land, cmip_progedmf_land
cpu-gpu: GPU ClimaAtmos with prognostic EDMF, GPU ClimaAtmos without EDMF, gpu_amip_progedmf, gpu_amip_progedmf_1M_land_he16, gpu_amip_progedmf_1M_land_he30, gpu_amip_progedmf_io
land: Global bucket simulation, Snowy Land (Optimal LAI), 19 years, Snowy Land + P Model, Snowy Land + P Model (Optimal LAI), Snowy Land, 19 years, Soil, Soil, 20 years


In [9]:
# Fixed-order categorical palette, validated for colour-vision deficiency
# on adjacent pairs. It has eight slots; a ninth config in a suite reuses
# the colours with dashed lines rather than inventing a hue.
PALETTE = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#6250d6",
    "#e34948",
]


def make_figure(suite: dict) -> go.Figure:
    """Speed, stability, and (if the suite has a leaderboard) accuracy."""
    has_accuracy = suite["accuracy_title"] is not None
    rows = 3 if has_accuracy else 2
    titles = [
        "Speed: throughput (SYPD)",
        "Stability: share of the configured run length simulated",
    ]
    if has_accuracy:
        titles.append(suite["accuracy_title"])
    fig = make_subplots(
        rows=rows,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08 if has_accuracy else 0.1,
        subplot_titles=titles,
    )
    suite_df = df[df["pipeline"].isin(suite["pipelines"])]
    suite_scores = scores[scores["pipeline"].isin(suite["pipelines"])]

    for i, run_name in enumerate(suite["run_names"]):
        sub = suite_df[suite_df["run_name"] == run_name].sort_values("date")
        color = PALETTE[i % len(PALETTE)]
        style = dict(
            legendgroup=run_name,
            mode="lines+markers",
            marker=dict(color=color, size=8),
            line=dict(
                color=color,
                width=2,
                dash="solid" if i < len(PALETTE) else "dash",
            ),
        )
        fig.add_trace(
            go.Scatter(
                x=sub["date"],
                y=sub["sypd"],
                name=run_name,
                customdata=sub[["job_state", "pipeline"]].fillna(""),
                hovertemplate=(
                    f"<b>{run_name}</b><br>"
                    "%{x|%Y-%m-%d}<br>SYPD: %{y:.3f}<br>"
                    "%{customdata[1]}<br>%{customdata[0]}<extra></extra>"
                ),
                **style,
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=sub["date"],
                y=sub["fraction_completed"] * 100,
                name=run_name,
                showlegend=False,
                customdata=sub[["job_state", "error"]]
                .fillna("")
                .assign(simulated_years=sub["simulated_years"]),
                hovertemplate=(
                    f"<b>{run_name}</b><br>"
                    "%{x|%Y-%m-%d}<br>%{y:.1f}% of target "
                    "(%{customdata[2]:.3g} simulated years)<br>"
                    "%{customdata[0]}<br>%{customdata[1]}<extra></extra>"
                ),
                **style,
            ),
            row=2,
            col=1,
        )
        # Mark genuine failures so a crash reads as a crash rather than just
        # a low value (see FAILURE_STATES above)
        failed = sub[sub["job_state"].isin(FAILURE_STATES)]
        fig.add_trace(
            go.Scatter(
                x=failed["date"],
                y=failed["fraction_completed"] * 100,
                name=run_name,
                legendgroup=run_name,
                showlegend=False,
                mode="markers",
                marker=dict(color=color, symbol="x", size=11),
                hoverinfo="skip",
            ),
            row=2,
            col=1,
        )
        if not has_accuracy:
            continue
        acc = suite_scores[suite_scores["run_name"] == run_name].sort_values(
            "date"
        )
        fig.add_trace(
            go.Scatter(
                x=acc["date"],
                y=acc["accuracy"],
                name=run_name,
                showlegend=False,
                customdata=acc[["job_state", "detail"]].fillna(""),
                hovertemplate=(
                    f"<b>{run_name}</b><br>"
                    "%{x|%Y-%m-%d}<br>Score: %{y:.3g}<br>"
                    "%{customdata[1]}<br>%{customdata[0]}<extra></extra>"
                ),
                **style,
            ),
            row=3,
            col=1,
        )

    # Reference lines for the coupler's relative score: the CMIP6 AMIP
    # median model is 1 by construction, and the best model's ratio is the
    # bar to beat. Drawn across the whole time axis so they read as
    # references.
    reference = suite_scores["reference"].dropna() if has_accuracy else []
    if len(reference):
        x_span = [suite_df["date"].min(), suite_df["date"].max()]
        for y, label in [
            (1.0, "CMIP6 AMIP median"),
            (reference.mean(), "Best CMIP6 AMIP run"),
        ]:
            fig.add_trace(
                go.Scatter(
                    x=x_span,
                    y=[y, y],
                    mode="lines+text",
                    line=dict(color="#8a8a85", width=1, dash="dot"),
                    text=["", label],
                    textposition="top left",
                    textfont=dict(color="#5f5f5a", size=11),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=3,
                col=1,
            )

    # Throughput spans two orders of magnitude across these configs
    # (slabplanet ~0.03 SYPD vs AMIP >2), so a linear axis hides structure
    fig.update_yaxes(title_text="SYPD", type="log", row=1, col=1)
    fig.update_yaxes(title_text="Length (%)", range=[0, 105], row=2, col=1)
    if has_accuracy:
        fig.update_yaxes(
            title_text=suite["accuracy_axis"], rangemode="tozero", row=3, col=1
        )
    fig.update_xaxes(title_text="Date", row=rows, col=1)
    fig.update_layout(
        height=900 if has_accuracy else 650,
        title=suite["title"],
        legend=dict(orientation="h", y=-0.1 if has_accuracy else -0.15),
    )
    return fig


os.makedirs("figures", exist_ok=True)
for suite in SUITES:
    fig = make_figure(suite)
    fig.write_json(f"figures/weekly-benchmarks-{suite['slug']}.json")
    fig.show()